In [1]:
from pathlib import Path

In [2]:
data_dir = Path("/home/rooneyish/Documents/Pulse_SLM/data")
train_file = data_dir / "train.txt"
val_file = data_dir / "val.txt"

In [3]:
train_txt = open(train_file).read()
val_txt = open(val_file).read()

In [4]:
import re

def clean_txt(text):
    # Remove the backslashes escaping the single quotes (\'s -> 's)
    text = text.replace("\\'", "'")
    
    # Fix the weird spacing around the text formatting
    text = re.sub(r"\s+'\s*([sn])", r"'\1", text)   # fixes ' s or ' n
    text = re.sub(r"={1,}\s*(.*?)\s*={1,}", r"\1", text) # removes headers
    text = re.sub(r"\s+([,.:;?!])", r"\1", text)    # fixes punctuation spacing
    text = re.sub(r"\s*@\s*-\s*@\s*", "-", text)    # fixes @-@ hyphens
    text = re.sub(r"\s+", " ", text).strip()        # collapses extra spaces
    
    return text

In [5]:
clean_train_txt = clean_txt(train_txt)
clean_val_txt = clean_txt(val_txt)

In [6]:
len(clean_train_txt), len(clean_val_txt)

(10521374, 1103839)

In [7]:
full_txt = clean_train_txt + clean_val_txt

In [8]:
len(full_txt)

11625213

In [9]:
import nltk.data

nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /home/rooneyish/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/rooneyish/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
from nltk.tokenize import sent_tokenize

sent_tokens = sent_tokenize(full_txt)

In [11]:
sent_tokens[:5]

['Valkyria Chronicles III Senjō no Valkyria 3: Unrecorded Chronicles ( Japanese: 戦場のヴァルキュリア3, lit.',
 'Valkyria of the Battlefield 3 ), commonly referred to as Valkyria Chronicles III outside Japan, is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable.',
 'Released in January 2011 in Japan, it is the third game in the Valkyria series.',
 'Employing the same fusion of tactical and real-time gameplay as its predecessors, the story runs parallel to the first game and follows the " Nameless ", a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven ".',
 'The game began development in 2010, carrying over a large portion of the work done on Valkyria Chronicles II.']

In [12]:
sent_w_SE = []
for each in sent_tokens:
    each = ''.join('<S> '+ each + ' <E>')
    sent_w_SE.append(each)

In [13]:
sent_w_SE[:5]

['<S> Valkyria Chronicles III Senjō no Valkyria 3: Unrecorded Chronicles ( Japanese: 戦場のヴァルキュリア3, lit. <E>',
 '<S> Valkyria of the Battlefield 3 ), commonly referred to as Valkyria Chronicles III outside Japan, is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable. <E>',
 '<S> Released in January 2011 in Japan, it is the third game in the Valkyria series. <E>',
 '<S> Employing the same fusion of tactical and real-time gameplay as its predecessors, the story runs parallel to the first game and follows the " Nameless ", a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven ". <E>',
 '<S> The game began development in 2010, carrying over a large portion of the work done on Valkyria Chronicles II. <E>']

In [14]:
tokens = []
for sentence in sent_w_SE:
    word = sentence.split()
    tokens.append(word)

In [15]:
len(tokens)

86251

In [16]:
corpus = {}
all_tokens = []
for each in tokens:
    for i in range(len(each)):
        all_tokens.append(each[i])

for each in all_tokens:
    if each in corpus:
        corpus[each] += 1
    else:
        corpus[each] = 1

corpus = {k:v for k, v in sorted(corpus.items(), key = lambda item: item[1], reverse = True)}
del corpus['<S>']
del corpus['<E>']
len(corpus)

134357

In [17]:
pad_token = "<PAD>"
unk_token = "<UNK>"
start_token = "<S>"
end_token = "<E>"

pad_id = 0
unk_id = 1
start_id = 2
end_id = 3

unique_words = list(corpus.keys())

In [18]:
stoi = {pad_token: pad_id, unk_token: unk_id, start_token: start_id, end_token: end_id}
itos = {pad_id: pad_token, unk_id: unk_token, start_id: start_token, end_id: end_token}

In [19]:
for idx, word in enumerate(unique_words):
    target_id = idx + 4
    stoi[word] = target_id
    itos[target_id] = word

In [20]:
len(stoi), len(itos)

(134361, 134361)

In [21]:
import torch

In [22]:
tokenized_sentence = [[stoi[token] for token in sentence if token in stoi] for sentence in tokens]

In [23]:
X_data = []
Y_data = []

In [24]:
max(map(len, tokenized_sentence))

564

In [25]:
tokenized_sentence[:2]

[[2,
  4021,
  5036,
  1132,
  25777,
  116,
  4021,
  4022,
  65766,
  5036,
  19,
  15730,
  65767,
  12934,
  3],
 [2,
  4021,
  5,
  4,
  29806,
  107,
  57,
  1744,
  798,
  8,
  13,
  4021,
  5036,
  1132,
  584,
  2900,
  20,
  9,
  7006,
  10505,
  266,
  117,
  398,
  18,
  7704,
  6,
  35945,
  16,
  4,
  1582,
  35946,
  3]]

In [26]:
sequence = []
for sentence in tokenized_sentence:
    for i in range(1, len(sentence)):
        sequence.append(sentence[:i+1])

In [27]:
sequence[:20]

[[2, 4021],
 [2, 4021, 5036],
 [2, 4021, 5036, 1132],
 [2, 4021, 5036, 1132, 25777],
 [2, 4021, 5036, 1132, 25777, 116],
 [2, 4021, 5036, 1132, 25777, 116, 4021],
 [2, 4021, 5036, 1132, 25777, 116, 4021, 4022],
 [2, 4021, 5036, 1132, 25777, 116, 4021, 4022, 65766],
 [2, 4021, 5036, 1132, 25777, 116, 4021, 4022, 65766, 5036],
 [2, 4021, 5036, 1132, 25777, 116, 4021, 4022, 65766, 5036, 19],
 [2, 4021, 5036, 1132, 25777, 116, 4021, 4022, 65766, 5036, 19, 15730],
 [2, 4021, 5036, 1132, 25777, 116, 4021, 4022, 65766, 5036, 19, 15730, 65767],
 [2,
  4021,
  5036,
  1132,
  25777,
  116,
  4021,
  4022,
  65766,
  5036,
  19,
  15730,
  65767,
  12934],
 [2,
  4021,
  5036,
  1132,
  25777,
  116,
  4021,
  4022,
  65766,
  5036,
  19,
  15730,
  65767,
  12934,
  3],
 [2, 4021],
 [2, 4021, 5],
 [2, 4021, 5, 4],
 [2, 4021, 5, 4, 29806],
 [2, 4021, 5, 4, 29806, 107],
 [2, 4021, 5, 4, 29806, 107, 57]]

In [28]:
max_len = max(map(len, sequence))
max_len

564

In [29]:
from torch.nn.utils.rnn import pad_sequence
tensor_seq = [torch.tensor(seq) for seq in sequence]
padded_sequence = pad_sequence(tensor_seq, batch_first=True, padding_side='left', padding_value=0)

In [32]:
padded_sequence[100]

tensor([    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [33]:
len(padded_sequence[100])

564

In [35]:
X = padded_sequence[:, :-1]
y = padded_sequence[:,-1]

In [36]:
X[0]

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [37]:
y[0]

tensor(4021)

In [39]:
num_samples = X.shape[0]
num_samples

2068115

In [40]:
split_idx = int(num_samples * 0.9)

In [41]:
X_train, y_train = X[:split_idx], y[:split_idx]
X_val, y_val = X[split_idx:], y[split_idx:]

In [43]:
len(X_train), len(y_train)

(1861303, 1861303)

In [45]:
len(X_val), len(y_val)

(206812, 206812)